# 01 - Raw data preprocessing
Purpose: split the wide multi-building table into one clean hourly series per building (`BUILDINGS` in `src/config.py`).

Outputs: `data/raw/<building>.csv` (timestamp, energy) and `data/interim/<building>_interim.csv` (hourly grid, missing hours removed).

Note: `scripts/Model_Training.py` does the same cleaning in code (`scripts/preprocessing.py`), so this notebook is for inspection.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
for p in (PROJECT_ROOT, PROJECT_ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.config import BUILDINGS, CONFIG, FEATURE_COLS

In [ ]:
import pandas as pd

SOURCE_PATH = PROJECT_ROOT / CONFIG["data_url"]
RAW_PATH = PROJECT_ROOT / "data" / "raw"
INTERIM_PATH = PROJECT_ROOT / "data" / "interim"
RAW_PATH.mkdir(parents=True, exist_ok=True)
INTERIM_PATH.mkdir(parents=True, exist_ok=True)

# Use relative paths (resolved from the project root) so the notebook is reproducible on other machines.
print(SOURCE_PATH)

### Split into one CSV per building

In [ ]:
wide = pd.read_csv(SOURCE_PATH, usecols=["timestamp"] + BUILDINGS)
wide["timestamp"] = pd.to_datetime(wide["timestamp"], utc=True)

for bld in BUILDINGS:
    out_df = wide[["timestamp", bld]].rename(columns={bld: "energy"})
    out_df.to_csv(RAW_PATH / f"{bld}.csv", index=False)
    print("Saved raw:", bld, out_df.shape)

### Preprocessing block (hourly grid, drop missing hours)

In [ ]:
for building in BUILDINGS:
    df = pd.read_csv(RAW_PATH / f"{building}.csv")
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").set_index("timestamp")

    # enforce hourly grid
    df_hr = df.resample("1h").sum(min_count=1)

    # remove missing hours
    df_valid = df_hr.loc[df_hr["energy"].notna()]

    print(building, "rows after cleaning:", df_valid.shape)
    df_valid.to_csv(INTERIM_PATH / f"{building}_interim.csv")